# Day 1 extra: summarize openai.com

`display_summary("https://openai.com")` fails with `week1/scraper.py` because the site is a JavaScript app. This notebook uses Playwright (a real Chromium) instead of `requests`.

One-time setup in the repo `.venv`:

```bash
uv pip install playwright
playwright install chromium
```

In [ ]:
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

from scraper_playwright import fetch_website_contents

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [ ]:
def messages_for(website: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]


async def summarize(url: str) -> str:
    website = await fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website),
    )
    return response.choices[0].message.content


async def display_summary(url: str):
    display(Markdown(await summarize(url)))

Preview the rendered text (first 500 characters) so you can see Playwright got past the empty `requests` scrape.

In [ ]:
preview = await fetch_website_contents("https://openai.com")
print(preview[:500])

In [ ]:
await display_summary("https://openai.com")